# 02 — Interactive Agent Tester & Quick Test
Test the vLLM endpoint live, try different game prompts, and compare output quality.

**Use this notebook to:**
- ✅ Sanity-check the vLLM server is working
- ✅ Test Thai / English language output
- ✅ Generate questions from any text snippet (no PDF needed)
- ✅ Compare question styles across game types
- ✅ Visualise question length & answer distribution from saved datasets

In [ ]:
import json, os, re
from pathlib import Path
from openai import OpenAI

BASE_URL = os.getenv("AI_BASE_URL", "http://localhost:8000/v1")
API_KEY  = os.getenv("AI_API_KEY",  "none")
MODEL    = os.getenv("AI_MODEL",    "Qwen/Qwen2.5-7B-Instruct-AWQ")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# ── Quick sanity check ────────────────────────────────────────────────────────
try:
    models = client.models.list()
    name = models.data[0].id if models.data else "unknown"
    print(f"✅ vLLM ready  |  model: {name}")
    print(f"   Base URL: {BASE_URL}")
except Exception as e:
    print(f"❌ Cannot connect to vLLM: {e}")
    print()
    print("Steps to start vLLM:")
    print("  1. ssh aip04@br1.paas.ku.ac.th")
    print("  2. module load slurm")
    print("  3. sbatch ~/ku_prep_arena/ai/scripts/slurm_vllm.sh")
    print("  4. squeue -u aip04   (wait for RUNNING)")
    print("  5. Look at log for node name: tail -f ~/ku_prep_arena/vllm-<JOB>.log")
    print("  6. On LOCAL: ssh -N -L 8000:dgx-XX:8000 aip04@br1.paas.ku.ac.th")
    raise

## 🧪 Quick Test — Thai Language Agent
ทดสอบว่า Qwen ตอบเป็นภาษาไทยได้ถูกต้อง

In [ ]:
NO_CHINESE = (
    "CRITICAL: Do NOT output Chinese (中文), Japanese, Korean, Arabic, or any script "
    "not present in the document. Write ONLY in the same language as the document "
    "(Thai = ภาษาไทย, English = English)."
)

GAME_PROMPTS = {
    "flappy":  f"Keep questions SHORT (under 12 words) and choices very brief (1-4 words). {NO_CHINESE}",
    "racer":   f"Keep EVERY question under 12 words. Choices 1-4 words each. {NO_CHINESE}",
    "shooter": f"Focus on IDENTIFICATION questions: 'Which term describes…?'. {NO_CHINESE}",
    "snake":   f"Prefer SEQUENTIAL questions: 'What is the FIRST step in…?'. {NO_CHINESE}",
    "bricks":  f"Focus on DEFINITIONS: 'What does X mean?'. {NO_CHINESE}",
}

def ask_questions(text: str, game_type: str = "shooter", n: int = 3) -> list[dict]:
    """Generate n questions from text using a specific game agent."""
    system = GAME_PROMPTS[game_type] + "\nRespond ONLY with a valid JSON array."
    user = f"""Create exactly {n} multiple-choice questions from this text.
CRITICAL: Write ONLY in the SAME language as the text (Thai or English). Do NOT use Chinese.
Return: [{{"id":1,"question":"...","choices":["A","B","C","D"],"correct":0,"explanation":"..."}}]

Text:
{text}"""
    res = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0.5, max_tokens=1024
    )
    raw = res.choices[0].message.content or "[]"
    cleaned = re.sub(r"^```[a-z]*\n?", "", raw, flags=re.I).strip()
    cleaned = re.sub(r"\n?```$", "", cleaned, flags=re.I).strip()
    return json.loads(cleaned)

# ── Test with Thai text ───────────────────────────────────────────────────────
thai_sample = """
กระบวนการสังเคราะห์แสงแบ่งออกเป็น 2 ขั้นตอนหลัก คือ ปฏิกิริยาที่ต้องการแสง (Light reactions) 
ซึ่งเกิดขึ้นที่เยื่อไทลาคอยด์ในคลอโรพลาสต์ และวัฏจักรคัลวิน (Calvin cycle) 
ซึ่งเกิดขึ้นในสโตรมา กระบวนการนี้ใช้พลังงานแสงแปลงคาร์บอนไดออกไซด์และน้ำ 
ให้เป็นกลูโคสและออกซิเจน
"""

print("📝 Testing with Thai text...")
print("─" * 60)
qs = ask_questions(thai_sample, game_type="shooter", n=3)
for q in qs:
    print(f"\nQ{q['id']}: {q['question']}")
    for i, c in enumerate(q['choices']):
        mark = "✅" if i == q['correct'] else "  "
        print(f"  {mark} {['A','B','C','D'][i]}) {c}")
    print(f"  💡 {q['explanation']}")

## Question length by game type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Question word count
df.groupby("game_type")["q_len"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="#336600", alpha=0.85
)
axes[0].set_title("Avg Question Length (words)")
axes[0].set_xlabel("Words")

# Choice word count
df.groupby("game_type")["choice_avg_len"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#f5c518", alpha=0.85
)
axes[1].set_title("Avg Choice Length (words)")
axes[1].set_xlabel("Words")

plt.tight_layout()
plt.savefig("../eval/results/agent_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: eval/results/agent_comparison.png")

## Answer distribution per game type

In [ ]:
pivot = df.groupby(["game_type", "correct"]).size().unstack(fill_value=0)
pivot.columns = ["A", "B", "C", "D"][:len(pivot.columns)]
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

ax = pivot_pct.plot(kind="bar", figsize=(10, 4), colormap="Set2", edgecolor="white")
ax.axhline(25, color="black", linestyle="--", linewidth=1, label="Ideal 25%")
ax.set_title("Answer Distribution by Game Type (%)")
ax.set_xlabel("")
ax.set_ylabel("Percentage (%)")
ax.legend(loc="upper right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../eval/results/answer_distribution.png", bbox_inches="tight")
plt.show()